# BBEH × PromptPotter

Runs PromptPotter's L1/L2/L3 optimization loop on BBEH and writes a `results_potter.json` next to `results_capo.json` / `results_dspy.json`.

**Methodology.** One global prompt is optimized on the full mini-BBEH pool (460 examples, pooled across all 23 tasks) and then evaluated on the held-out non-mini set (~4,060 examples, i.e. every BBEH example NOT in mini). The mini/non-mini partition is a HuggingFace-native flag on each record, so the train/test split is disjoint by construction — zero leakage. This matches how BBEH is officially graded: one model, one prompt, per-task accuracies reported for the harmonic-mean metric.

Not a per-task loop. Specialising a different prompt per task inflates the score relative to what you'd actually deploy, and with 20 mini examples per task the per-task optimizer hits noise-level 100% and early-stops on round 1 without learning anything.

**Runs locally against this repo** (unlike the Colab-based CAPO/DSPy notebooks). Prereqs:

- `pip install -e ".[dev,jupyter]"` from the repo root
- `datasets` package: `pip install datasets`
- `.env` with the relevant provider key (`OPENROUTER_API_KEY` for the default mistral-small target / optimizer; `GROQ_API_KEY` if you swap `datasets/bbeh/campaign.json` to gpt-oss-120b)
- A running PromptPotter-compatible backend at `http://127.0.0.1:8000` exposing an `llm_only` node. BBEH's `datasets/bbeh/pipeline.json` constrains the active pipeline to that single node, so every query is a plain LLM call with the optimized prompt as system message.

**Configuration is shared with the CLI.** `datasets/bbeh/pipeline.json` defines the target pipeline (model, provider, reasoning effort) and `datasets/bbeh/campaign.json` carries every loop-control knob plus the optimizer LLM. Edit those files to retune; the notebook reads them as-is. The values there are unmeasured starting points — a later sweep will replace them.

**Before running the full campaign, smoke-test first**: `python scripts/smoke_campaign.py --dataset bbeh` (~90s).

In [ ]:
# Cell 1 — env + autoreload + path setup
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

# Notebook lives in `notebooks/`. Repo root is one level up. `shared_config.py`
# and the sibling `results_*.json` files stay in `docs/research/bbeh-comparison/`
# because the CAPO / DSPy comparison notebooks live there too.
_REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
_BBEH_DIR = _REPO_ROOT / "docs" / "research" / "bbeh-comparison"
if str(_BBEH_DIR) not in sys.path:
    sys.path.insert(0, str(_BBEH_DIR))

try:
    from dotenv import load_dotenv
    load_dotenv(_REPO_ROOT / ".env")
except ImportError:
    pass

# Provider key matches whatever `datasets/bbeh/campaign.json::optimizer_llm.provider`
# (and `pipeline.json::llm_only.config.provider`) say. Default config = openrouter.
assert os.environ.get("OPENROUTER_API_KEY") or os.environ.get("GROQ_API_KEY"), (
    "Need OPENROUTER_API_KEY (default mistral-small) or GROQ_API_KEY in env"
)
print("env OK")

In [ ]:
# Cell 2 — load BBEH data
from shared_config import SPLIT_SEED, load_and_split

train_pool, test_by_task = load_and_split()
tasks = sorted(test_by_task.keys())

# Sanity: mini/non-mini partition must be disjoint.
_train_keys = {(ex["input"], ex["target"]) for ex in train_pool}
_test_keys = {
    (ex["input"], ex["target"])
    for items in test_by_task.values()
    for ex in items
}
assert not (_train_keys & _test_keys), "LEAK: train and test overlap"

n_test = sum(len(v) for v in test_by_task.values())
print(
    f"Loaded BBEH: {len(train_pool)} train (mini, pooled), "
    f"{n_test} test (non-mini, per-task) across {len(tasks)} tasks (seed={SPLIT_SEED})"
)

In [ ]:
# Cell 3 — run the full campaign
#
# Defaults live in datasets/bbeh/{pipeline,campaign}.json — same files
# `python -m promptpotter new bbeh` reads. Edit those to retune for both
# CLI and notebook. The kwargs below are notebook-only ad-hoc overrides:
# uncomment any line to shadow campaign.json for this run; leave commented
# to inherit. Variable names are kept visible for tab-completion and so
# you don't have to look up the field schema.
#
# What to watch for after the run (rasch-validation smoking gun):
#   1. Active cycle dir → `cat .promptpotter/active_session.json`
#   2. hard_samples_campaign.json::sample_order → non-empty; rasch.converged true
#   3. log.md::## Hard Samples → heatmap with ≥4 distinct δ_s values
#   4. output.log → HIT/MISS lines now carry #NNN sample IDs (sample_id plumbing fixed)
#
# Ctrl+C is safe at any time:
#   - Optimization loop catches the interrupt, finalizes (writes log.md +
#     index.json + heatmap), and the runner returns None.
#   - The httpx connection pool is closed unconditionally via try/finally,
#     so the kernel won't get stuck on "Connecting to kernel" next time.
#   - The ~4060-query test-eval phase is skipped on interrupt — re-run cell 3
#     to resume optimization, or call run_optimization_notebook again to
#     just retry test-eval against the saved winner.
from bbeh_potter_runner import run_bbeh_campaign

await run_bbeh_campaign(
    train_pool,
    test_by_task,
    output_path=_BBEH_DIR / "results_potter.json",
    # max_rounds=8,
    # n_variants=5,
    # sp_budget_ttest=20,
    # optimizer_model="openai/gpt-oss-120b",
    # optimizer_provider="groq",
)

## Interpretation

`results_potter.json` now sits next to `results_capo.json` and `results_dspy.json` (when those have been run) with an identical top-level schema. The `config.note` field flags that PromptPotter's hyperparameters here are untuned — any head-to-head number below should be read as a floor, not a ceiling, for PromptPotter on BBEH.

Next steps:
- Hyperparameter sweep over `MAX_ROUNDS`, `N_VARIANTS`, `SP_BUDGET_TTEST`.
- Feed the three `results_*.json` files into `docs/research/related-work.md` for the comparison table.